# Merinos Endüstriyel AI — Day 28 (Faz 4 Kapanışı / Capstone)
## Uçtan Uca Endüstriyel Hibrit RAG Hattı ve Canlıya Geçiş Kalite Kapısı

**Şirket:** Merinos Halı Sanayi ve Ticaret A.Ş. (Gaziantep 4. OSB)  
**Geliştirici:** Seydi Eryılmaz (@seydivakkas)  
**Telif Hakkı:** (c) 2026 Seydi Eryılmaz. ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR.  

--- 
### 🎯 Amaç ve Kapsam
Faz 4 boyunca geliştirdiğimiz tüm bilgi getirme ve arama bileşenlerini (Markdown chunking, Okapi BM25, Qdrant HNSW + Int8 SQ, RRF $k=60$ füzyon, Cross-Encoder re-ranking ve Ragas değerlendirme metrikleri) tek bir kurumsal üretim hattında birleştirmek ve üretim ortamı dağıtımı için otomatik kalite kapısını (Deployment Gate) devreye almaktır.

### 📌 Adım 1: Ortam Kurulumu ve Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import json
import time
from pathlib import Path
import matplotlib.pyplot as plt

from day28.mini_project.src.models import DocumentItem, QueryRequest
from day28.mini_project.src.document_indexer import DocumentIndexer
from day28.mini_project.src.hybrid_retriever import HybridRetriever
from day28.mini_project.src.generator_llm import GroundedGenerator
from day28.mini_project.src.deployment_gate import DeploymentGate
from day28.mini_project.src.visualizer import plot_capstone_diagnostic_panel

print("✅ Gerekli modüller ve Merinos Faz 4 kütüphaneleri başarıyla yüklendi.")

### 📌 Adım 2: Merinos Fabrika Teknik Doküman Korpusunun İncelenmesi

In [ ]:
corpus_path = Path("mini_project/fixtures/merinos_factory_corpus.json")
with open(corpus_path, "r", encoding="utf-8") as f:
    raw_docs = json.load(f)

documents = [DocumentItem(**d) for d in raw_docs]
print(f"📚 Toplam Yüklenen Doküman Sayısı: {len(documents)}")
for doc in documents[:3]:
    print(f"  • [{doc.doc_id}] {doc.title} | Birim: {doc.department} | Makine: {doc.machine}")

### 📌 Adım 3: Markdown Başlık Hiyerarşisine Duyarlı Parçalama ve Breadcrumbs Enjeksiyonu

In [ ]:
indexer = DocumentIndexer(collection_name="merinos_capstone_kb")
chunks = indexer.chunk_documents(documents)
print(f"🧩 Üretilen Toplam Anlamsal Parça (Chunk) Sayısı: {len(chunks)}")

sample_chunk = chunks[1]
print(f"\nÖrnek Parça ID: {sample_chunk.chunk_id}")
print(f"Hiyerarşi Zinciri (Breadcrumbs): {sample_chunk.breadcrumbs}")
print(f"Metin İçeriği:\n{sample_chunk.text[:200]}...")

### 📌 Adım 4: Okapi BM25 Seyrek (Sparse) İndeksinin İnşa Edilmesi

In [ ]:
# BM25 indeksi parçalar üzerinden eğitilir
indexer.bm25_index.fit(chunks)
vocab_size = len(indexer.bm25_index.doc_freqs)
print(f"📖 BM25 Kelime Dağarcığı (Vocabulary Size): {vocab_size} benzersiz terim")

# Basit leksikal test sorgusu
sample_scores = indexer.bm25_index.score_all("atkı ipliği pnömatik fren gerginliği")
best_idx = int(sample_scores.index(max(sample_scores)))
print(f"En Yüksek BM25 Skoru: {max(sample_scores):.4f} -> Parça: {chunks[best_idx].chunk_id}")

### 📌 Adım 5: Bellek İçi Qdrant HNSW İndeksi ve INT8 Skalar Kuantalama

In [ ]:
# Hem BM25 hem Qdrant indekslerini tam olarak inşa et
indexer.build_indexes(documents)
coll_info = indexer.qdrant_client.get_collection("merinos_capstone_kb")
print(f"⚡ Qdrant Vektör Koleksiyonu: merinos_capstone_kb")
print(f"  • İndekslenen Vektör Sayısı: {coll_info.points_count}")
print(f"  • Vektör Boyutu: {coll_info.config.params.vectors.size}")
print(f"  • Mesafe Metriği: {coll_info.config.params.vectors.distance}")

### 📌 Adım 6: Ön-Filtreleme ve Reciprocal Rank Fusion (RRF $k=60$) Birleşimi

In [ ]:
retriever = HybridRetriever(indexer, rrf_k=60, sparse_weight=0.4, dense_weight=0.6)

test_query = "Van de Wiele tezgahında atkı tel kopuşunda ne yapılır?"
req = QueryRequest(query=test_query, department_filter="dokuma_salonu_1", top_k=5, enable_reranking=False)
candidates, lats = retriever.retrieve(req)

print(f"🔍 Ön-Filtreleme ve RRF Sıralaması Sonuçları (Top-{len(candidates)}):")
for c in candidates:
    print(f"  • [Sıra {c.rank}] {c.chunk_id} | RRF Skoru: {c.rrf_score:.5f} | BM25: {c.sparse_score:.2f} | Dense: {c.dense_score:.4f}")

### 📌 Adım 7: Cross-Encoder Derin Yeniden Sıralama (Re-Ranking)

In [ ]:
req_rerank = QueryRequest(query=test_query, department_filter="dokuma_salonu_1", top_k=3, enable_reranking=True)
candidates_reranked, lats_reranked = retriever.retrieve(req_rerank)

print(f"🎯 Cross-Encoder Sonrası Nihai Sıralama:")
for c in candidates_reranked:
    print(f"  • [{c.chunk_id}] Re-rank Skoru: {c.rerank_score:.4f} | Bölüm: {c.breadcrumbs}")

### 📌 Adım 8: Alıntı Destekli (Grounded) Cevap Üretimi

In [ ]:
generator = GroundedGenerator()
response = generator.generate(req_rerank, candidates_reranked, lats_reranked)

print("=" * 70)
print("🤖 SENTEZLENEN TEKNİK MODEL CEVABI:")
print("=" * 70)
print(response.answer)
print(f"\n📚 Alıntılanan Referanslar: {', '.join(response.citations)}")
print(f"⚡ Toplam Uçtan Uca Gecikme: {response.total_latency_ms:.2f} ms")

### 📌 Adım 9: Otomatik Canlıya Geçiş Kalite Kapısı (Deployment Gate Audit)

In [ ]:
gate = DeploymentGate(retriever, generator)
queries_path = Path("mini_project/fixtures/capstone_queries.json")
with open(queries_path, "r", encoding="utf-8") as f:
    test_queries = json.load(f)

gate_result = gate.run_gate_audit(test_queries[:10])
print("🛡️ DEPLOYMENT GATE RAPORU:")
print(f"  • Sadakat (Faithfulness)         : %{gate_result.faithfulness * 100:.1f}")
print(f"  • Bağlamsal Kesinlik (Precision) : %{gate_result.context_precision * 100:.1f}")
print(f"  • Bağlamsal Kapsama (Recall)     : %{gate_result.context_recall * 100:.1f}")
print(f"  • Cevap Uygunluğu (Relevance)    : %{gate_result.answer_relevance * 100:.1f}")
print(f"  • Harmonik Ragas Skoru           : %{gate_result.ragas_composite * 100:.1f}")
print(f"  • Canlı Dağıtım Kararı           : {'✅ ONAYLANDI' if gate_result.passed else '❌ REDDEDİLDİ'}")

### 📌 Adım 10: 2x2 Master Teşhis Paneli Görselleştirmesi & Kapanış

In [ ]:
report_path = Path("mini_project/outputs/capstone_benchmark_report.json")
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        bench_data = json.load(f)
    fig = plot_capstone_diagnostic_panel(bench_data)
    plt.show()
else:
    print("Lütfen önce CLI üzerinden 'benchmark --plot' komutunu çalıştırınız.")